In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
# ==========================================
# BNPL DATA INGESTION
# ==========================================

raw_csv = "/Volumes/workspace/default/bnpl_raw_csv/nigerian_bnpl_full.xls"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_csv)
)

# Standardise date fields for temporal analysis
df = (
    df
    .withColumn("purchase_date", F.to_timestamp("purchase_date"))
    .withColumn("first_payment_due", F.to_timestamp("first_payment_due"))
)

print("BNPL dataset loaded successfully.")

BNPL dataset loaded successfully.


%md
# 01. BNPL Data Audit

## Objective

This notebook performs the initial audit of the synthetic Nigerian BNPL transaction dataset before downstream data engineering, feature construction and predictive modelling.

The audit evaluates:

1. Dataset scale and schema
2. Data completeness
3. Transaction and customer uniqueness
4. Target prevalence
5. Temporal coverage
6. Numerical distributions
7. Key transaction-level risk relationships
8. Customer behavioural signals
9. Data validity
10. Potential synthetic-data artefacts

## Primary Target

The primary supervised-learning target is `default_90d`.

The primary modelling unit is the **BNPL transaction/loan**, because the default outcome is attached to each transaction. For a transaction occurring at time `t`:

- Predictive features must represent information available at or before `t`
- The target represents the future `default_90d` outcome

Customer-level historical features used later in the project must therefore be constructed from transactions occurring strictly before the current transaction.

## Role of This Audit

This notebook establishes whether the dataset is structurally suitable for downstream modelling and identifies relationships that may materially affect model interpretation.

## Data Limitation

The dataset is synthetic. Therefore, the distributions, default rates and relationships identified in this notebook describe the supplied synthetic portfolio only. They must **not** be interpreted as empirical estimates of actual Nigerian BNPL default behaviour.

In [0]:
# ==========================================
# DATASET SCALE AND SCHEMA
# ==========================================

row_count = df.count()
column_count = len(df.columns)

print("Number of rows:", row_count)
print("Number of columns:", column_count)

print("\nColumn names:")
for col in df.columns:
    print("-", col)

print("\nData types:")
df.printSchema()

Number of rows: 2000000
Number of columns: 16

Column names:
- transaction_id
- purchase_date
- customer_id
- merchant_category
- merchant_name
- customer_state
- principal_ngn
- interest_rate_monthly
- tenor_days
- num_installments
- provider
- credit_score
- first_time_customer
- first_payment_due
- default_30d
- default_90d

Data types:
root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: timestamp (nullable = tru

In [0]:
# ==========================================
# SAMPLE RECORDS
# ==========================================

display(df.limit(10))

transaction_id,purchase_date,customer_id,merchant_category,merchant_name,customer_state,principal_ngn,interest_rate_monthly,tenor_days,num_installments,provider,credit_score,first_time_customer,first_payment_due,default_30d,default_90d
BNPL-0000274330,2024-07-10T00:00:00.000Z,CUS-00036437,groceries,Groceries Store 105,Oyo,35416.989134742886,0.0,14,1,FairMoney,645,false,2024-08-09T00:00:00.000Z,false,false
BNPL-0001873807,2023-08-01T00:00:00.000Z,CUS-00131005,furniture,Furniture Store 820,Lagos,43154.84670603445,0.0,30,1,FairMoney,716,true,2023-08-31T00:00:00.000Z,false,false
BNPL-0001029126,2022-02-12T00:00:00.000Z,CUS-00648656,electronics,Electronics Store 346,Abuja (FCT),247235.6151124731,0.0,14,1,Branch,470,false,2022-03-14T00:00:00.000Z,false,false
BNPL-0001512791,2023-01-26T00:00:00.000Z,CUS-00611557,furniture,Furniture Store 393,Lagos,70679.21723036664,0.03272774744299827,30,1,Branch,620,false,2023-02-25T00:00:00.000Z,false,false
BNPL-0000552055,2023-01-12T00:00:00.000Z,CUS-00499951,fashion,Fashion Store 956,Ekiti,225083.5043127901,0.04965523147852287,90,3,Branch,536,false,2023-02-11T00:00:00.000Z,false,false
BNPL-0001320205,2024-10-31T00:00:00.000Z,CUS-00361857,groceries,Groceries Store 902,Enugu,68583.77260243788,0.0,30,1,Branch,700,true,2024-11-30T00:00:00.000Z,false,false
BNPL-0001240716,2022-07-14T00:00:00.000Z,CUS-00017739,fashion,Fashion Store 304,Lagos,26826.99512408854,0.0395323781705534,30,1,Branch,735,false,2022-08-13T00:00:00.000Z,false,false
BNPL-0001905652,2024-12-18T00:00:00.000Z,CUS-00607469,fashion,Fashion Store 747,Adamawa,11537.404930539655,0.02708009892409436,30,1,Renmoney,587,true,2025-01-17T00:00:00.000Z,false,false
BNPL-0000756202,2024-08-05T00:00:00.000Z,CUS-00655563,other,Other Store 325,Cross River,25579.23439870545,0.04543871068905091,30,1,Carbon,661,false,2024-09-04T00:00:00.000Z,false,false
BNPL-0000173103,2022-06-29T00:00:00.000Z,CUS-00535542,electronics,Electronics Store 46,Osun,34657.09419332149,0.036495570934379086,14,1,FairMoney,683,false,2022-07-29T00:00:00.000Z,false,false


In [0]:
# ==========================================
# CORE DATA QUALITY AUDIT
# ==========================================

# 1. Missing values
missing_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

print("MISSING VALUES")
display(missing_df)


# 2. Key integrity and explicit duplicate audit
print("KEY INTEGRITY")

total_rows = df.count()
unique_transactions = df.select("transaction_id").distinct().count()
unique_customers = df.select("customer_id").distinct().count()

duplicate_transaction_ids = (
    df.groupBy("transaction_id")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Total rows:", total_rows)
print("Unique transaction IDs:", unique_transactions)
print("Unique customers:", unique_customers)
print("Duplicate transaction IDs:", duplicate_transaction_ids)


# 3. Target distribution
print("\nTARGET DISTRIBUTION")

target_df = df.agg(
    F.count("*").alias("total_transactions"),
    F.sum(F.col("default_30d").cast("int")).alias("default_30d_count"),
    F.avg(F.col("default_30d").cast("double")).alias("default_30d_rate"),
    F.sum(F.col("default_90d").cast("int")).alias("default_90d_count"),
    F.avg(F.col("default_90d").cast("double")).alias("default_90d_rate")
)

display(target_df)


# 4. Date coverage
print("DATE COVERAGE")

date_df = df.agg(
    F.min("purchase_date").alias("first_purchase"),
    F.max("purchase_date").alias("last_purchase"),
    F.countDistinct(F.to_date("purchase_date")).alias("active_days")
)

display(date_df)


# 5. Numerical summary
print("NUMERICAL VARIABLES")

display(
    df.select(
        "principal_ngn",
        "interest_rate_monthly",
        "tenor_days",
        "num_installments",
        "credit_score"
    ).summary()
)

MISSING VALUES


transaction_id,purchase_date,customer_id,merchant_category,merchant_name,customer_state,principal_ngn,interest_rate_monthly,tenor_days,num_installments,provider,credit_score,first_time_customer,first_payment_due,default_30d,default_90d
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


KEY INTEGRITY
Total rows: 2000000
Unique transaction IDs: 2000000
Unique customers: 633356
Duplicate transaction IDs: 0

TARGET DISTRIBUTION


total_transactions,default_30d_count,default_30d_rate,default_90d_count,default_90d_rate
2000000,100001,0.0500005,160001,0.0800005


DATE COVERAGE


first_purchase,last_purchase,active_days
2022-01-01T00:00:00.000Z,2024-12-30T00:00:00.000Z,1095


NUMERICAL VARIABLES


summary,principal_ngn,interest_rate_monthly,tenor_days,num_installments,credit_score
count,2000000,2000000,2000000,2000000,2000000
mean,49978.009310041736,0.026248329580360687,40.312311,1.4501685,619.510411
stddev,46533.201140625184,0.01821959465681942,23.017800953802244,0.6689066043370873,84.75804932615274
min,5000.0,0.0,14,1,300
25%,21155.061582928513,0.0,30,1,562
50%,36313.594001194746,0.032148834113267286,30,1,620
75%,62308.43863164352,0.04107098957278667,60,2,677
max,500000.0,0.049999996661038465,90,3,850


%md
## Core Audit Interpretation

The dataset contains **2.0 million BNPL transactions across 633,356 customers and 16 variables**.

The transaction IDs are unique, and the explicit duplicate-ID audit confirms that no transaction ID occurs more than once. This provides a clean transaction-level foundation for downstream modelling.

No missing values are observed across the dataset. While this simplifies feature engineering, complete absence of missingness should be treated as a characteristic of the synthetic dataset rather than an assumption about production BNPL data.

The dataset covers approximately three years, from **January 2022 through December 2024**, providing sufficient temporal depth for chronological train, validation and out-of-time evaluation.

The primary target, `default_90d`, occurs at approximately **8%**, while `default_30d` occurs at approximately **5%**. The near-exact target prevalence is itself indicative of a controlled synthetic data-generation process.

The approximately 8% `default_90d` prevalence should be used as the baseline when interpreting classification metrics, particularly PR-AUC.

The numerical variables show meaningful variation in transaction exposure, repayment tenor and credit quality. Principal is positively skewed, while credit scores span the conventional 300 to 850 range.

Overall, the dataset passes the initial structural audit and is suitable for progression to point-in-time feature engineering and predictive modelling.

In [0]:
# ==========================================
# CREDIT SCORE AND DEFAULT DIAGNOSTIC
# ==========================================

score_bands = (
    df
    .withColumn(
        "score_band",
        F.when(F.col("credit_score") < 500, "<500")
         .when(F.col("credit_score") < 550, "500-549")
         .when(F.col("credit_score") < 600, "550-599")
         .when(F.col("credit_score") < 650, "600-649")
         .when(F.col("credit_score") < 700, "650-699")
         .when(F.col("credit_score") < 750, "700-749")
         .otherwise("750+")
    )
    .groupBy("score_band")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(F.col("default_90d").cast("int")).alias("defaults_90d"),
        F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d")
    )
    .orderBy("score_band")
)

display(score_bands)

score_band,transactions,defaults_90d,default_rate_90d
500-549,251880,49578,0.19683182467841828
550-599,403429,78795,0.19531317778345136
600-649,461971,105,2.2728699420526398E-4
650-699,377867,71,1.878967996676079E-4
700-749,220722,32,1.449787515517257E-4
750+,126397,22,1.7405476395800532E-4
<500,157734,31398,0.19905663965917303


%md
## Credit Score Diagnostic: Interpretation

The credit-score analysis identifies the strongest and most unusual relationship in the dataset.

Transactions with credit scores below 600 have observed `default_90d` rates of approximately **19.5% to 19.9%**, whereas transactions with scores of 600 or above have default rates close to zero.

The sharp discontinuity around a score of 600 is substantially stronger than a gradual credit-risk relationship would normally suggest. It is therefore treated as a **synthetic-data artefact** and not as evidence of an actual Nigerian BNPL credit-score threshold.

This finding has an important modelling implication.

The project will retain `credit_score` in the **Full Information model**, because it represents an available conventional credit-risk variable. However, a separate **Behavioural model** will exclude raw credit-score information and focus on BNPL transaction and historical behavioural information.

This comparison is necessary to determine how much predictive performance is attributable to conventional score information versus BNPL behavioural information.

The observed magnitude of the relationship must not be generalised beyond the supplied synthetic dataset.

In [0]:
# ==========================================
# KEY TRANSACTION-LEVEL RISK DIAGNOSTICS
# ==========================================

print("DEFAULT RATE BY FIRST-TIME CUSTOMER")

display(
    df.groupBy("first_time_customer")
      .agg(
          F.count("*").alias("transactions"),
          F.sum(F.col("default_90d").cast("int")).alias("defaults_90d"),
          F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d")
      )
      .orderBy("first_time_customer")
)


print("DEFAULT RATE BY TENOR")

display(
    df.groupBy("tenor_days")
      .agg(
          F.count("*").alias("transactions"),
          F.sum(F.col("default_90d").cast("int")).alias("defaults_90d"),
          F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d")
      )
      .orderBy("tenor_days")
)


print("DEFAULT RATE BY PRINCIPAL BAND")

principal_bands = (
    df.withColumn(
        "principal_band",
        F.when(F.col("principal_ngn") < 25000, "<25K")
         .when(F.col("principal_ngn") < 50000, "25K-49K")
         .when(F.col("principal_ngn") < 100000, "50K-99K")
         .otherwise("100K+")
    )
    .groupBy("principal_band")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(F.col("default_90d").cast("int")).alias("defaults_90d"),
        F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d")
    )
    .orderBy("principal_band")
)

display(principal_bands)

DEFAULT RATE BY FIRST-TIME CUSTOMER


first_time_customer,transactions,defaults_90d,default_rate_90d
false,1200048,24261,0.020216691332346708
true,799952,135740,0.16968518111086664


DEFAULT RATE BY TENOR


tenor_days,transactions,defaults_90d,default_rate_90d
14,399093,29139,0.0730130571069901
30,900489,65745,0.07301033105346096
60,500499,36073,0.07207407007806209
90,199919,29044,0.1452788379293614


DEFAULT RATE BY PRINCIPAL BAND


principal_band,transactions,defaults_90d,default_rate_90d
100K+,206014,46823,0.22728067024571147
25K-49K,669993,42631,0.06362902298979244
50K-99K,483258,30347,0.0627966841728435
<25K,640735,40200,0.06274044651845147


## Transaction-Level Risk Diagnostics: Interpretation

Three transaction-level variables show meaningful separation of observed default risk in the synthetic portfolio.

### First-Time Customer Status

First-time customers show an observed `default_90d` rate of approximately **17.0%**, compared with approximately **2.0%** for repeat customers.

This represents a strong synthetic signal and supports retaining customer lifecycle status as an eligible predictor.

### Tenor

Default rates are broadly similar across 14-, 30- and 60-day transactions, while 90-day transactions show a materially higher default rate of approximately **14.5%**.

Longer-tenor transactions therefore represent a higher-risk portion of this synthetic portfolio and are relevant for subsequent risk interpretation and stress testing.

### Principal Exposure

Default rates are relatively stable across the lower principal bands but increase sharply for transactions of **₦100,000 or more**, where the observed default rate is approximately **22.7%**.

This makes transaction exposure relevant not only to PD modelling but also to later EAD, segmentation and portfolio stress analysis.

All three relationships are treated as characteristics of the synthetic dataset. They should not be interpreted as empirical Nigerian BNPL risk estimates.

In [0]:
# ==========================================
# POINT-IN-TIME CUSTOMER BEHAVIOURAL AUDIT
# ==========================================

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_date", "transaction_id")
)

behavior_audit = (
    df
    .withColumn(
        "previous_default_90d",
        F.lag(F.col("default_90d").cast("int")).over(customer_window)
    )
    .withColumn(
        "previous_transaction",
        F.lag("purchase_date").over(customer_window)
    )
    .withColumn(
        "days_since_previous",
        F.datediff(
            F.to_date("purchase_date"),
            F.to_date("previous_transaction")
        )
    )
)

print("DEFAULT RATE BY PREVIOUS TRANSACTION DEFAULT STATUS")

previous_default_analysis = (
    behavior_audit
    .filter(F.col("previous_default_90d").isNotNull())
    .groupBy("previous_default_90d")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(F.col("default_90d").cast("int")).alias("current_defaults"),
        F.avg(F.col("default_90d").cast("double")).alias("current_default_rate")
    )
    .orderBy("previous_default_90d")
)

display(previous_default_analysis)

DEFAULT RATE BY PREVIOUS TRANSACTION DEFAULT STATUS


previous_default_90d,transactions,current_defaults,current_default_rate
0,1257251,100353,0.07981938371892328
1,109393,8816,0.08059016573272512


In [0]:
# ==========================================
# CUSTOMER HISTORY SIGNAL AUDIT
# ==========================================

history_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_date", "transaction_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)

behavior_features_audit = (
    df
    .withColumn(
        "prior_transaction_count",
        F.count("*").over(history_window)
    )
    .withColumn(
        "prior_default_count",
        F.sum(F.col("default_90d").cast("int")).over(history_window)
    )
    .withColumn(
        "prior_exposure",
        F.sum("principal_ngn").over(history_window)
    )
)

repeat_customer_audit = (
    behavior_features_audit
    .filter(F.col("prior_transaction_count") > 0)
    .withColumn(
        "history_band",
        F.when(F.col("prior_transaction_count") == 1, "1 prior")
         .when(F.col("prior_transaction_count") <= 3, "2-3 prior")
         .when(F.col("prior_transaction_count") <= 5, "4-5 prior")
         .otherwise("6+ prior")
    )
    .groupBy("history_band")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(F.col("default_90d").cast("int")).alias("defaults_90d"),
        F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d"),
        F.avg("prior_exposure").alias("avg_prior_exposure"),
        F.avg("prior_default_count").alias("avg_prior_defaults")
    )
)

print("DEFAULT RATE BY PRIOR CUSTOMER HISTORY")

display(
    repeat_customer_audit.orderBy("history_band")
)

DEFAULT RATE BY PRIOR CUSTOMER HISTORY


history_band,transactions,defaults_90d,default_rate_90d,avg_prior_exposure,avg_prior_defaults
1 prior,533861,42535,0.07967429724216603,49984.03947238742,0.0803935855962507
2-3 prior,619591,49520,0.07992369159655321,118952.81437207908,0.189967252590822
4-5 prior,178895,14284,0.07984571955616424,216106.49154833134,0.34613041169401043
6+ prior,34297,2830,0.08251450564189287,323388.1533942427,0.5164300084555501


In [0]:
# ==========================================
# RECENCY AND HISTORICAL DEFAULT SIGNAL
# ==========================================

behavior_recency = (
    df
    .withColumn(
        "previous_purchase_date",
        F.lag("purchase_date").over(customer_window)
    )
    .withColumn(
        "days_since_previous",
        F.datediff(
            F.to_date("purchase_date"),
            F.to_date("previous_purchase_date")
        )
    )
    .withColumn(
        "prior_transaction_count",
        F.count("*").over(history_window)
    )
    .withColumn(
        "prior_default_count",
        F.sum(F.col("default_90d").cast("int")).over(history_window)
    )
    .withColumn(
        "prior_default_rate",
        F.when(
            F.col("prior_transaction_count") > 0,
            F.col("prior_default_count") /
            F.col("prior_transaction_count")
        )
    )
)

recency_analysis = (
    behavior_recency
    .filter(F.col("days_since_previous").isNotNull())
    .withColumn(
        "recency_band",
        F.when(F.col("days_since_previous") <= 7, "0-7 days")
         .when(F.col("days_since_previous") <= 30, "8-30 days")
         .when(F.col("days_since_previous") <= 90, "31-90 days")
         .otherwise("90+ days")
    )
    .groupBy("recency_band")
    .agg(
        F.count("*").alias("transactions"),
        F.avg(F.col("default_90d").cast("double")).alias("default_rate_90d"),
        F.avg("prior_transaction_count").alias("avg_prior_transactions"),
        F.avg("prior_default_rate").alias("avg_prior_default_rate")
    )
)

print("DEFAULT RATE BY RECENCY OF PREVIOUS TRANSACTION")

display(
    recency_analysis.orderBy("recency_band")
)

DEFAULT RATE BY RECENCY OF PREVIOUS TRANSACTION


recency_band,transactions,default_rate_90d,avg_prior_transactions,avg_prior_default_rate
0-7 days,40338,0.08143685854529228,2.4934553026922504,0.08045893334083105
31-90 days,264124,0.08022746891611515,2.418576123336009,0.08005273299349205
8-30 days,117686,0.0798990534133202,2.480490457658515,0.07982109058681507
90+ days,944496,0.07971553082278802,2.086428105571649,0.08012569426156556


%md
## Behavioural History and Recency: Interpretation

The behavioural audit uses customer-specific chronological windows ordered by `purchase_date` and `transaction_id`. Historical features are calculated using rows strictly preceding the current transaction.

This is important because future transactions and future outcomes must not contribute to the information available at the prediction point.

### Previous Default

The observed current default rate is approximately **8.0%** regardless of whether the immediately preceding transaction was a default.

Therefore, the simple previous-default indicator provides limited standalone separation in this synthetic dataset.

This should not be interpreted as evidence that customer history is irrelevant. It indicates only that this individual historical indicator has limited marginal separation when considered alone.

### Prior Transaction History

Customers with more historical transactions have substantially higher cumulative prior exposure. However, current default rates remain broadly close to the portfolio baseline.

This indicates that simple historical transaction count and cumulative exposure have limited univariate discriminatory power in the supplied data.

### Recency

Default rates remain broadly stable across the recency bands, at approximately 8%.

Simple recency therefore provides limited standalone separation of `default_90d`. It remains relevant as part of the broader behavioural feature set because customer activity patterns may become informative when combined with other variables.

These findings support the project's required comparison between a conventional Full Information model and a Behavioural model based on BNPL history and transaction behaviour.

In [0]:
# ==========================================
# DATA VALIDITY AUDIT
# ==========================================

validity_checks = df.select(
    F.sum(
        (F.col("principal_ngn") <= 0).cast("int")
    ).alias("nonpositive_principal"),

    F.sum(
        (F.col("interest_rate_monthly") < 0).cast("int")
    ).alias("negative_interest"),

    F.sum(
        (F.col("tenor_days") <= 0).cast("int")
    ).alias("invalid_tenor"),

    F.sum(
        (F.col("num_installments") <= 0).cast("int")
    ).alias("invalid_installments"),

    F.sum(
        (
            (F.col("credit_score") < 300) |
            (F.col("credit_score") > 850)
        ).cast("int")
    ).alias("invalid_credit_score"),

    F.sum(
        (
            F.col("first_payment_due") < F.col("purchase_date")
        ).cast("int")
    ).alias("payment_due_before_purchase"),

    F.sum(
        (
            F.col("default_30d") &
            ~F.col("default_90d")
        ).cast("int")
    ).alias("default_30_without_90"),

    F.sum(
        F.col("transaction_id").isNull().cast("int")
    ).alias("null_transaction_id"),

    F.sum(
        F.col("customer_id").isNull().cast("int")
    ).alias("null_customer_id")
)

display(validity_checks)

nonpositive_principal,negative_interest,invalid_tenor,invalid_installments,invalid_credit_score,payment_due_before_purchase,default_30_without_90,null_transaction_id,null_customer_id
0,0,0,0,0,0,0,0,0


# Final Audit Assessment

## Overall Status: PASS

The BNPL dataset passes the initial structural and analytical audit and is suitable for progression to downstream Big Data engineering and predictive modelling.

### Key Findings

| Audit Area | Finding | Assessment |
|---|---|---|
| Dataset scale | 2.0M transactions across 633K+ customers | PASS |
| Transaction uniqueness | No duplicate transaction IDs | PASS |
| Missingness | No missing values across 16 variables | PASS |
| Target | `default_90d` prevalence ≈ 8% | PASS |
| Temporal coverage | January 2022 to December 2024 | PASS |
| Numerical validity | Variables fall within expected ranges | PASS |
| Date validity | No payment dates before purchase dates | PASS |
| Target consistency | No `default_30d = 1` without `default_90d = 1` | PASS |
| Point-in-time history | Historical windows exclude the current and future transactions | PASS |
| Behavioural diagnostics | Historical and recency signals audited | PASS |
| Synthetic-data artefacts | Several unusually strong relationships identified | DOCUMENTED |

## Major Analytical Findings

The strongest observed relationships are associated with:

- Credit score
- First-time customer status
- High transaction principal
- 90-day tenor

In contrast, simple historical indicators such as previous default, transaction count and recency show substantially weaker standalone separation.

The particularly strong discontinuity in default rates around a credit score of 600 is treated as a synthetic-data artefact. It is retained for the Full Information modelling specification but motivates the separate Behavioural model robustness experiment.

## Modelling Implications

The audit supports the following downstream design:

1. `default_90d` remains the primary supervised-learning target.
2. The primary modelling unit remains the transaction/loan.
3. Customer historical features must use only transactions strictly preceding the prediction transaction.
4. A chronological modelling design should be used rather than a random train-test split.
5. Full Information and Behavioural model specifications should be evaluated separately.
6. Credit score should remain available to the Full Information model but should not dominate the overall project narrative.
7. Behavioural features should be evaluated independently of raw credit-score information.
8. The synthetic nature of the dataset must remain explicit throughout model evaluation, segmentation, stress testing and business recommendations.

## Conclusion

The dataset is structurally clean, temporally suitable and sufficiently rich for the proposed BNPL credit-risk analytics workflow.

No fundamental data-quality issue prevents progression to the next stage.

The principal limitation is not data validity but **synthetic-data realism**. Consequently, all subsequent model performance metrics and observed risk relationships should be interpreted as demonstrations of the proposed credit-risk methodology rather than empirical estimates of Nigerian BNPL credit behaviour.

The next stage is therefore to transform the audited transaction data into the Bronze, Silver and Gold analytical layers while preserving the point-in-time feature requirement established in this audit.